## Introduction to prompting

Users ask questions like "show me popular Python repos from last month". The GitHub API needs language:python created:>2024-10-29 sort:stars.
We'll work with the GitHub API as our example, translating requests like "show me popular Python repos from last month" into properly formatted query strings with the right parameters, filters, and syntax.
This notebook shows how to build prompts that make this translation reliable. We'll cover parameter extraction, handling ambiguous requests, and preventing common failures.

Let's load environment variables and initialize the OpenAI client. The `call_gpt()` function sends prompts to the model with optional system instructions and returns the response text.

In [ ]:
from openai import OpenAI
import json
import os
from dotenv import load_dotenv

load_dotenv('/workspace/.env', override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
def call_gpt(prompt: str, system_prompt: str | None = None) -> str:
    messages = [{"role": "user", "content": prompt}]
    if system_prompt:
        messages.insert(0, {"role": "system", "content": system_prompt})
    
    response = client.chat.completions.create(
        model="o4-mini-2025-04-16",
        messages=messages
    )
    return response.choices[0].message.content

### Zero-shot and few-shot prompting

We'll start with zero-shot prompting — giving the model a task with no examples. Then we'll add examples (few-shot) to improve consistency.

Ask the model to convert the request directly, specifying only the output format:

In [ ]:
user_request = "Find Python repositories created in 2024 with over 1000 stars"

zero_shot_prompt = f"""Convert this request into GitHub API search parameters:
"{user_request}"

Return as JSON with these fields:
- q: the search query string
- sort: sort field (stars, forks, updated)
- order: asc or desc
"""

response = call_gpt(zero_shot_prompt)
print(response)

The model correctly extracts the key components: language filter, star threshold, and date range. It also makes reasonable assumptions — sorting by stars in descending order — since the request implied we want the most popular repos. The model wrapped the JSON in markdown code fences, which will break if you try to parse it directly with `json.loads()`. You'll need to strip the backticks and json label before parsing. 

Now we provide examples showing input-output pairs. This teaches the model the expected format and demonstrates how to handle different query types:

In [ ]:
user_request_few_shot = "Find TypeScript repositories from 2024 with more than 500 stars and active issues"
few_shot_prompt = f"""Convert natural language requests into GitHub API search parameters.

Examples:

Request: "Find popular JavaScript repos"
Output: {{
  "q": "language:javascript",
  "sort": "stars",
  "order": "desc"
}}

Request: "Show me repos about machine learning updated this month"
Output: {{
  "q": "topic:machine-learning pushed:>2024-10-01",
  "sort": "updated",
  "order": "desc"
}}

Request: "Python projects with good first issues"
Output: {{
  "q": "language:python label:good-first-issue state:open",
  "sort": "created",
  "order": "desc"
}}


Now convert this request: {user_request_few_shot}


Output:"""

response = call_gpt(few_shot_prompt)
print(response)

### Chain-of-thought 

Chain-of-thought makes the model show its reasoning before answering. Ask the model to reason through ambiguous terms before generating the query. "Trending" could mean recent stars, activity, or creation date. "Beginner-friendly" might map to labels like good-first-issue or documentation tags. Explicit reasoning reduces inconsistent interpretations.

In [ ]:
user_request_cot = "Find trending AI repositories that are beginner-friendly"
cot_prompt = f"""Convert this request into GitHub API search parameters:
{user_request_cot}

Think step-by-step:
1. What does "trending" mean? (recent activity, stars, etc.)
2. How do we identify "AI" repositories?
3. What makes a repo "beginner-friendly"?
4. What filters should we combine?

Then provide the final JSON output."""

response = call_gpt(cot_prompt)
print(response)

### Structured output and validation

Asking for JSON in the prompt doesn't guarantee you'll get parseable, schema-compliant output. The model might add markdown, rearrange fields, or return malformed JSON.

In [ ]:
user_query_structured = "Show me the best React libraries from last year"
prompt_only = f"""Convert this request into GitHub API search parameters:
{user_query_structured}

Return ONLY valid JSON with this structure:
{{
  "query_params": {{...}},
  "reasoning": "...",
  "confidence": "high|medium|low"
}}
"""

response = call_gpt(prompt_only)
# No guarantee this is valid JSON or follows the schema
print(response)

OpenAI's structured output mode guarantees the response matches your schema. Define the structure with Pydantic, and the API will only return valid objects — no markdown wrapping, no missing fields, no type mismatches.

In [ ]:
from pydantic import BaseModel
from typing import Literal, List

# Define the schema using Pydantic
class QueryParams(BaseModel):
    q: str
    sort: Literal["stars", "forks", "updated", "created"]
    order: Literal["asc", "desc"]
    per_page: int = 30

class GitHubQueryResponse(BaseModel):
    query_params: QueryParams
    reasoning: str
    potential_issues: List[str]
    confidence: Literal["high", "medium", "low"]

def call_gpt_structured(
    prompt: str, 
    response_format: Type[BaseModel], 
    system_prompt: str | None = None
) -> BaseModel:
    messages = [{"role": "user", "content": prompt}]
    if system_prompt:
        messages.insert(0, {"role": "system", "content": system_prompt})
    
    response = client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",  # Must use a model that supports structured outputs
        messages=messages,
        response_format=response_format
    )
    return response.choices[0].message.parsed

In [ ]:
user_request = "Show me the best React libraries from last year"
structured_prompt = f"""Convert this request into GitHub API search parameters:
"{user_request}"

Consider:
- What does "best" mean?
- How to filter for "last year"?
- Flag any ambiguities or assumptions"""

result = call_gpt_structured(structured_prompt, GitHubQueryResponse)

# result is now a GitHubQueryResponse object, guaranteed to match schema
print(f"Query: {result.query_params.q}")
print(f"Sort: {result.query_params.sort}")
print(f"Confidence: {result.confidence}")
print(f"Issues: {result.potential_issues}")

# Can also convert to dict
print(json.dumps(result.model_dump(), indent=2))

In [ ]:
# Test with an ambiguous request
ambiguous_request = "Find good repos for learning web development"

print("=" * 60)
print("Without structured output:")
print("=" * 60)

prompt = f"""Convert to GitHub API params: "{ambiguous_request}"
Return JSON with: query_params, reasoning, potential_issues, confidence"""

unstructured_response = call_gpt(prompt)
print(unstructured_response)
print("\nNotice: May have inconsistent format, extra text, invalid JSON\n")

print("=" * 60)
print("With structured output:")
print("=" * 60)

structured_response = call_gpt_structured(
    f"""Convert this request into GitHub API search parameters: "{ambiguous_request}"
    
    Think about:
    - What makes a repo "good for learning"?
    - What does "web development" encompass?
    - What filters capture this?"""
, GitHubQueryResponse)

print(json.dumps(structured_response.model_dump(), indent=2))

In [ ]:
from pydantic import ValidationError, field_validator

class StrictQueryParams(BaseModel):
    q: str
    sort: Literal["stars", "forks", "updated", "created"]  # Only these values allowed
    order: Literal["asc", "desc"]
    per_page: int = 30
    
    # Can add custom validation
    @field_validator('q')
    def q_not_empty(cls, v):
        if not v or v.strip() == "":
            raise ValueError("Query string cannot be empty")
        return v

valid_request = "Find Python repos"
result = call_gpt_structured(
    f"Convert: {valid_request}",
    response_format=StrictQueryParams  # Using strict schema
)
print("Valid:", result.model_dump())

# If the LLM tries to return invalid data, OpenAI's API will retry automatically
# or raise an error if it can't satisfy the schema

### Handling edge cases and ambiguity

Users type "find some good python stuff" or "show me trending repos". The prompts need to surface ambiguity and document assumptions, not guess.

#### Example 1: Vague and ambiguous queries

In [ ]:
user_query_ambiguous = "find some good python stuff"
prompt = f"""Convert this user request to GitHub API parameters:
{user_query_ambiguous}

This query is vague. In your response:
1. Identify ALL ambiguous terms (e.g., "good", "stuff")
2. Document your interpretation of each term
3. List alternative interpretations the user might have meant
4. Set confidence level appropriately

Make reasonable assumptions but be transparent about them."""

print("Prompt that surfaces ambiguity:")
print("=" * 60)
response = call_gpt_structured(prompt, GitHubQueryResponse)
print(json.dumps(response.model_dump(), indent=2))

#### Example 2: Conflicting requirements

User asks for "new projects with lots of commit history" - these might conflict. The following prompt explicitly asks LLM to detect conflicts, explain prioritization decisions, and document trade-offs:

In [ ]:
conflicting_request = "Find new projects with lots of commit history"

conflict_aware_prompt = f"""Convert this user request to GitHub API parameters:
"{conflicting_request}"

IMPORTANT: Check for logical conflicts between requirements.
If you find conflicts:
- Clearly identify them in potential_issues
- Explain which requirement you prioritized and why
- Suggest what the user might actually want

Be explicit about trade-offs made."""

print("Handling conflicting requirements:")
print("=" * 60)
result = call_gpt_structured(conflict_aware_prompt, GitHubQueryResponse)
print(json.dumps(result.model_dump(), indent=2))

The prompt forced the LLM to:
   1. Recognize the conflict
   2. Make a defensible choice
   3. Document it for the user

#### Example 3: Impossible or invalid constraints

User asks for something GitHub API doesn't support (e.g., "repos with most downloads")

In [ ]:
invalid_request = "Find Python repos with the most downloads this week"

# Prompt with API constraint awareness
api_aware_prompt = f"""Convert this user request to GitHub API parameters:
"{invalid_request}"

CRITICAL: GitHub API search supports these filters:
- language, stars, forks, size, created, pushed, topics, license
- Does NOT support: downloads, npm/pip installs, weekly metrics

If the request asks for unsupported features:
1. Flag this clearly in potential_issues
2. Suggest the closest supported alternative
3. Explain why it's only an approximation
4. Set confidence to 'low' if using poor proxy metrics"""

print("Handling impossible constraints:")
print("=" * 60)
result = call_gpt_structured(api_aware_prompt, GitHubQueryResponse)
print(json.dumps(result.model_dump(), indent=2))

Without this, the LLM might:
   - Silently use 'stars' as a proxy for 'downloads'
   - Not warn the user about the limitation
   - Return results that don't match user intent     

Provide explicit list of supported/unsupported features so LLM can flag impossible requests and suggest alternatives.

#### Best practices for prompts

Key techniques demonstrated above:

In [ ]:
def build_prompt(user_request: str) -> str:
    return f"""Convert this user request to GitHub API search parameters:
"{user_request}"

CONTEXT - GitHub API Capabilities:
- Searchable: language, stars, forks, size, created/pushed dates, topics, license, issues
- NOT searchable: downloads, page views, active users, external metrics
- Sort options: stars, forks, updated, created
- Time filters: Use created:>YYYY-MM-DD or pushed:>YYYY-MM-DD format

YOUR TASK:
1. Parse the user's intent from potentially vague language
2. Map requirements to available GitHub filters
3. Identify ambiguous terms (e.g., "good", "popular", "recent")
4. Flag conflicts (e.g., "new" + "mature", "simple" + "feature-rich")
5. Note impossible constraints and suggest alternatives
6. Document ALL assumptions in the reasoning field
7. List issues in potential_issues array
8. Set confidence based on ambiguity level:
   - high: Clear, supported requirements
   - medium: Some ambiguity but reasonable interpretation
   - low: Heavy assumptions or unsupported features

Make the best query possible, but be transparent about limitations."""

# Test with multiple edge cases
test_cases = [
    "find good python stuff",
    "trending ML repos for beginners",
    "repos with most npm downloads",
    "new projects with stable APIs"
]

print("Prompt in action:")
print("=" * 60)
for test_query in test_cases:
    print(f"\nQuery: \"{test_query}\"")
    print("-" * 60)
    result = call_gpt_structured(build_prompt(test_query), GitHubQueryResponse)
    print(f"Search: {result.query_params.q}")
    print(f"Confidence: {result.confidence}")
    if result.potential_issues:
        print(f"Issues: {', '.join(result.potential_issues[:2])}...")  # Show first 2
    print()

### Key Takeaways

1. **Specify constraints**: Tell the model what the API can't do, not just what you want

2. **Force transparency**: Use `reasoning` and `potential_issues` fields to surface assumptions

3. **Calibrate confidence**: Make the model assess certainty based on how ambiguous the request is

4. **Surface errors early**: Flag unclear requests instead of guessing

5. **Layer validation**: 
   - Structured outputs catch schema violations
   - Prompt engineering catches semantic problems

**The pattern:**
```
Bad: "Convert X to Y"
Good: "Convert X to Y. Flag ambiguities, check for conflicts with API constraints, 
      document assumptions, set confidence level"
```